In [0]:
df = spark.read.table("circuitbox.pii.customer_orders_lakeflow_csv")
df.select('payment_mode').distinct().display()

In [0]:
df.createOrReplaceTempView("vw_payment_modes")

In [0]:
%sql
CREATE OR REPLACE TABLE circuitbox.pii.payment_modes_managers AS
select distinct
  payment_mode,
  case
    when payment_mode = 'Cash on Delivery' then 'gitesh.sahare@accenture.com'
    when payment_mode = 'Credit Card' then 'ashish.bv.singh@accenture.com'
    when payment_mode = 'Debit Card' then 'sreeja.sarangpur@accenture.com'
    when payment_mode = 'Net Banking' then 'gitesh.sahare@accenture.com'
    when payment_mode = 'Wallet' then 'mohammad.b.sameer@accenture.com'
    else 'ashish.bv.singh@accenture.com'
  end as Payment_mode_manager
from
  vw_payment_modes

In [0]:
%sql
select * from circuitbox.pii.payment_modes_managers


In [0]:
%sql
CREATE OR REPLACE TABLE circuitbox.pii.silver_payment_info
AS 
select a.*,b.Payment_mode_manager
from circuitbox.pii.customer_orders_lakeflow_csv a
left join circuitbox.pii.payment_modes_managers b on a.payment_mode=b.payment_mode

In [0]:
%sql
select * from circuitbox.pii.payment_modes_managers

BUILDING A FUNCTION WHERE PAYMENT MANAGERS CAN ONLY SEE THEIR DATA

In [0]:
%sql
CREATE OR REPLACE FUNCTION circuitbox.pii.ispayment_manager(par_payment_mode STRING)
  RETURNS BOOLEAN
  LANGUAGE SQL
  RETURN  exists (
    Select
      1
    from
      circuitbox.pii.payment_modes_managers A
    WHERE
      A.payment_mode = ispayment_manager.par_payment_mode
      and A.Payment_mode_manager = current_user()
  )

Apply RLS


In [0]:
%sql
ALTER TABLE circuitbox.pii.silver_payment_info
SET ROW FILTER circuitbox.pii.ispayment_manager ON (payment_mode);


In [0]:
%sql
select * from circuitbox.pii.silver_payment_info